In [7]:
#Delete for submission
import os
from pathlib import Path

# Automatically searches for the CSV files from your home directory
csv_files = list(Path.home().rglob('Resale Flat Prices*.csv'))

if csv_files:
    os.chdir(csv_files[0].parent)
    print(f"Found files in: {os.getcwd()}")
else:
    print("CSV files not found!")

Found files in: C:\Users\Andre\Downloads


<pre style="border: 1px solid #ccc; padding: 10px;">
<h1>Library Importing</h1>
</pre>

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import glob
import warnings
import datetime as dt
warnings.filterwarnings('ignore')


plt.rcParams['figure.figsize'] = (11, 5)
sns.set_style('whitegrid')


print("✅ Libraries loaded")

✅ Libraries loaded


<pre style="border: 1px solid #ccc; padding: 10px;">
<h1>Data Exploration & Cleaning</h1>
</pre>

## Loading & Exploring

`Load & Explore real Singapore housing data`

### Load & Inspect

In [9]:
# Load the data from multiple CSV files
files = glob.glob('Resale Flat Prices*.csv')
print(f"Found {len(files)} CSV files: {files}")
hdb_raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

Found 1 CSV files: ['Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv']


EmptyDataError: No columns to parse from file

In [ ]:
# Inspect data shape, types, and preview after loading

print("📊 Data Overview:\n")
print(f"Total rows loaded: {len(hdb_raw):,}")
print(f"Total columns:     {hdb_raw.shape[1]}")

print("\nData types:")
print(hdb_raw.dtypes)
print("\nFirst 5 rows:")
print(hdb_raw.head())

📊 Data Overview:

Total rows loaded: 972,105
Total columns:     11

Data types:
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
resale_price           float64
remaining_lease         object
dtype: object

First 5 rows:
     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  ANG MO KIO    3 ROOM   216  ANG MO KIO AVE 1     04 TO 06   

   floor_area_sqm      flat_model  lease_commence_date  resale_price  \
0            31.0        IMPROVED      

---
## Data Cleaning

`Clean & Process messy real-world data professionally.`

### Filter Data

In [ ]:
# Convert 'month' column from string to datetime format
hdb_raw['month'] = pd.to_datetime(hdb_raw['month'], format='%Y-%m')

# Apply chosen date range filter (2012-2023)
start_date = '2012-01-01'
end_date = '2023-12-31'

# Filter the dataset to include only records within the specified date range
hdb = hdb_raw[(hdb_raw['month'] >= start_date) & (hdb_raw['month'] <= end_date)].copy()

# Display the number of rows after applying the date filter and explain the reason for choosing this range
print(f"After date filter ({start_date} to {end_date}): {len(hdb):,} rows")
print(f"Reason we chose this range: This 12-year period was chosen because the dataset from 2012 onwards uses a consistent registration-date recording system, which reduces inconsistencies and simplifies data cleaning and preprocessing. It also captures a complete housing market cycle, including the peak before cooling measures, the stabilisation period, and the sharp resale price increase after COVID-19. Using recent years ensures the data is relevant, consistent, and large enough for meaningful analysis without unnecessary complexity.")

After date filter (2012-01-01 to 2023-12-31): 261,694 rows
Reason we chose this range: This 12-year period was chosen because the dataset from 2012 onwards uses a consistent registration-date recording system, which reduces inconsistencies and simplifies data cleaning and preprocessing. It also captures a complete housing market cycle, including the peak before cooling measures, the stabilisation period, and the sharp resale price increase after COVID-19. Using recent years ensures the data is relevant, consistent, and large enough for meaningful analysis without unnecessary complexity.


### Drop Duplicates

In [ ]:
# Identify number of rows in dataset before removing duplicates
rows_before = len(hdb)

# Remove duplicating rows in dataset
hdb.drop_duplicates(inplace=True)

# Identify number of rows in dataset after removing duplicates
rows_after = len(hdb)

# Display the number of rows removed and the percentage of duplicates in the dataset
duplicates_removed = rows_before - rows_after
percentage_duplicates = (duplicates_removed / rows_before) * 100
print(f"Duplicates removed: {duplicates_removed:,} rows")
print(f"Percentage of duplicates: {percentage_duplicates:.2f}%")
print(f"Remaining rows after removing duplicates: {rows_after:,} rows")

Duplicates removed: 554 rows
Percentage of duplicates: 0.21%
Remaining rows after removing duplicates: 261,140 rows


### Handle Missing Values

In [ ]:
# Confirm missing values in filtered dataset
print("Missing values in each column:")
print(hdb.isnull().sum())

Missing values in each column:
month                      0
town                       0
flat_type                  0
block                      0
street_name                0
storey_range               0
floor_area_sqm             0
flat_model                 0
lease_commence_date        0
resale_price               0
remaining_lease        55142
dtype: int64


In [ ]:
# Drop or impute missing values using the 5% threshold rule (drop if missing values < 5%, impute with median if missing values >= 5%)
missing_threshold = 0.05 * len(hdb)

# Parse through "remaining_lease" column to convert string values to datetime, handling non-numeric entries by coercing them to NaN
hdb['remaining_lease'] = pd.to_datetime(hdb['remaining_lease'], errors='coerce')

for column in hdb.columns:
    missing_count = hdb[column].isnull().sum()
    if missing_count > 0:
        if missing_count < missing_threshold:
            hdb.dropna(subset=[column], inplace=True)
            print(f"Dropped {missing_count} rows with missing values in '{column}' (below 5% threshold).")
        else:
            median_value = hdb[column].median()
            hdb[column].fillna(median_value, inplace=True)
            print(f"Imputed {missing_count} missing values in '{column}' with median value (above 5% threshold).")

# Display the number of rows remaining after handling missing values
print(f"Rows after handling missing values: \n{len(hdb):,} rows\n")
print(f"Remaining missing values: \n{hdb.isnull().sum()}\n")

Imputed 224011 missing values in 'remaining_lease' with median value (above 5% threshold).
Rows after handling missing values: 
261,140 rows

Remaining missing values: 
month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
resale_price           0
remaining_lease        0
dtype: int64



### Remove Outliers

In [ ]:
# Using IQR method to identify and remove outliers in 'resale_price' column
Q1 = hdb['resale_price'].quantile(0.25)
Q3 = hdb['resale_price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
hdb = hdb[(hdb['resale_price'] >= lower) & (hdb['resale_price'] <= upper)]

# Using IQR method to identify and remove outliers in 'floor_area_sqm' column
Q1 = hdb['floor_area_sqm'].quantile(0.25)
Q3 = hdb['floor_area_sqm'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
hdb = hdb[(hdb['floor_area_sqm'] >= lower) & (hdb['floor_area_sqm'] <= upper)]

print(f"Outlier removal method: IQR Method")
print(f"Rows removed: {rows_before - len(hdb)}")
print(f"Rows remaining: {len(hdb):,}")
print(f"\nResale price range after cleaning:")
print(f"Min: ${hdb['resale_price'].min():,.0f}")
print(f"Max: ${hdb['resale_price'].max():,.0f}")
print(f"\nFloor area range after cleaning:")
print(f"Min: {hdb['floor_area_sqm'].min():,.0f} sqm")
print(f"Max: {hdb['floor_area_sqm'].max():,.0f} sqm")

Outlier removal method: IQR Method
Rows removed: 8153
Rows remaining: 253,541

Resale price range after cleaning:
Min: $140,000
Max: $858,137

Floor area range after cleaning:
Min: 31 sqm
Max: 169 sqm


### Parse Lease Data

In [ ]:
# Define a parsing function to convert 'remaining_lease' text to numeric years, handling various formats and missing values
def parse_lease_years(text):
    """
    Converts remaining_lease text to numeric years.
    '63 years 01 month'  → 63.08
    '45 years'           → 45.0
    '70 years 06 months' → 70.5
    """
    if pd.isna(text):
        return np.nan
    text = str(text).strip()
    years = 0
    months = 0
    if 'year' in text:
        years = int(text.split('year')[0].strip())
    if 'month' in text:
        months = int(text.split('month')[0].split()[-1].strip())
    return round(years + months / 12, 2)


# Test it:
test_cases = ['63 years 01 month', '45 years', '70 years 06 months', '99 years', None]
print("Function test:")
for t in test_cases:
    print(f"  '{t}' → {parse_lease_years(t)}")

Function test:
  '63 years 01 month' → 63.08
  '45 years' → 45.0
  '70 years 06 months' → 70.5
  '99 years' → 99.0
  'None' → nan


In [ ]:
# Apply the parsing function to the 'remaining_lease' column and create a new column 'remaining_lease_years'
hdb['remaining_lease_years'] = hdb['remaining_lease'].apply(parse_lease_years)

# Check results of parsing function
print("remaining_lease_years stats:")
print(hdb['remaining_lease_years'].describe().round(1))

# Create lease brackets for analysis; grouping data into '<50 years', '50-69 years', '70-89 years', '90+ years' based on remaining lease years
def lease_bracket(years):
    if pd.isna(years):
        return 'Unknown'
    elif years < 50:
        return '< 50 years'
    elif years < 70:
        return '50–69 years'
    elif years < 90:
        return '70–89 years'
    else:
        return '90+ years'
    
# Apply the lease_bracket function to create a new column 'lease_bracket'
hdb['lease_bracket'] = hdb['remaining_lease_years'].apply(lease_bracket)
print("\nLease bracket distribution:")
print(hdb['lease_bracket'].value_counts())

remaining_lease_years stats:
count    253541.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: remaining_lease_years, dtype: float64

Lease bracket distribution:
lease_bracket
< 50 years    253541
Name: count, dtype: int64


### Feature Engineering

In [ ]:
# Feature 1: price_per_sqm (resale price divided by floor area in sqm)
hdb['price_per_sqm'] = hdb['resale_price'] / hdb['floor_area_sqm']

# Feature 2: year
hdb['year'] = hdb['month'].dt.year

# Feature 3: region
town_to_region = {
    'BISHAN': 'Central', 'BUKIT MERAH': 'Central', 'BUKIT TIMAH': 'Central',
    'CENTRAL AREA': 'Central', 'GEYLANG': 'Central', 'KALLANG/WHAMPOA': 'Central',
    'MARINE PARADE': 'Central', 'QUEENSTOWN': 'Central', 'TOA PAYOH': 'Central',
    'BEDOK': 'East', 'PASIR RIS': 'East', 'TAMPINES': 'East',
    'SEMBAWANG': 'North', 'WOODLANDS': 'North', 'YISHUN': 'North',
    'ANG MO KIO': 'North-East', 'HOUGANG': 'North-East', 'PUNGGOL': 'North-East',
    'SENGKANG': 'North-East', 'SERANGOON': 'North-East',
    'BUKIT BATOK': 'West', 'BUKIT PANJANG': 'West', 'CHOA CHU KANG': 'West',
    'CLEMENTI': 'West', 'JURONG EAST': 'West', 'JURONG WEST': 'West',
}
hdb['region'] = hdb['town'].map(town_to_region)

# Check unmapped towns
unmapped_towns = hdb[hdb['region'].isna()]['town'].unique()
if len(unmapped_towns) > 0:
    print(f"⚠️  Add these towns to town_to_region: {', '.join(unmapped_towns)}")

# Display new columns created
print("Feature engineering complete. New columns:")
for col in ['price_per_sqm', 'year', 'region', 'remaining_lease_years', 'lease_bracket']:
     if col in hdb.columns:
         print(f"  ✅ {col}")
     else:
         print(f"  ❌ {col} — NOT YET CREATED")

Feature engineering complete. New columns:
  ✅ price_per_sqm
  ✅ year
  ✅ region
  ✅ remaining_lease_years
  ✅ lease_bracket


<pre style="border: 1px solid #ccc; padding: 10px;">
<h1>Exploratory Analysis</h1>
</pre>

## Which town offers the best value?

> A first-time buyer has a $400,000 budget. Where should they buy?

In [ ]:
# Calculate town rankings
town_value = (
    hdb.groupby('town')
    .agg(
        Median_Price_per_sqm = ('price_per_sqm', 'median'),
        Median_Resale_Price  = ('resale_price',  'median'),
        Transactions         = ('resale_price',  'count')
    )
    .round(0)
    .sort_values('Median_Price_per_sqm')
    .reset_index()
)

# Key metrics 


### 📝 Q1 Written Findings

---

**Finding:**  
Among the [___] towns analysed, **[Town]** offers the best value for buyers with a $400,000 budget, with a median price per sqm of **$[___]**. In contrast, **[Town]** is the most expensive at **$[___]/sqm** — approximately **[___]×** more expensive per square metre.

Towns in the **[region]** region consistently rank as the most affordable on a per-sqm basis, while **Central** region towns command the highest prices.

**Business Recommendation:**  
First-time buyers with a $400,000 budget will get approximately **[___] sqm** in [cheapest town] compared to only **[___] sqm** in [most expensive town]. We recommend buyers prioritise towns such as **[Town 1]**, **[Town 2]**, and **[Town 3]** for the best value. *[Note: this analysis does not account for proximity to MRT, school zones, or flat condition.]*

---
## How much does lease depreciation cost?

> Does a flat with a 70-year lease really cost more than a 40-year lease?

---
## Are prices going up or down?

> Is HDB a good long-term investment?

In [ ]:
EFJWEJKSFJKFJKWF

---
## Any geographic differences?

> Does living in the "Central" region command a premium?

In [ ]:
# ── Q4: Regional analysis ─────────────────────────────────────
region_stats = (
    hdb.groupby('region')
    .agg(
        Median_Price         = ('resale_price',  'median'),
        Median_Price_per_sqm = ('price_per_sqm', 'median'),
        Mean_Price           = ('resale_price',  'mean'),
        Transactions         = ('resale_price',  'count')
    )
    .round(0)
    .sort_values('Median_Price', ascending=False)
    .reset_index()
)

print('Q4 Results: Resale Price by Region')
print('=' * 60)
print(region_stats.to_string(index=False))

# ── Key insight numbers ────────────────────────────────────────
central_row   = region_stats[region_stats['region'] == 'Central']
cheapest_row  = region_stats.iloc[-1]

if not central_row.empty:
    central_price  = central_row['Median_Price'].values[0]
    cheapest_price = cheapest_row['Median_Price']
    premium_abs    = central_price - cheapest_price
    premium_pct    = (central_price / cheapest_price - 1) * 100
    print(f'\n💡 Central median price:        ${central_price:,.0f}')
    print(f'💡 Cheapest region ({cheapest_row["region"]}): ${cheapest_price:,.0f}')
    print(f'💡 Central premium:             +${premium_abs:,.0f} ({premium_pct:.1f}% higher)')

In [ ]:
# ── Q4: Side-by-side chart ────────────────────────────────────
# Left: raw median price  |  Right: price per sqm (size-adjusted)

region_order_price  = region_stats.sort_values('Median_Price', ascending=False)['region'].tolist()
region_order_ppsqm  = region_stats.sort_values('Median_Price_per_sqm', ascending=False)['region'].tolist()

def get_region_colors(order):
    return [REGION_COLORS.get(r, C_GREY) for r in order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Median Resale Price ─────────────────────────────────
ax = axes[0]
rs_price = region_stats.set_index('region').reindex(region_order_price)
colors_l = get_region_colors(region_order_price)
bars = ax.bar(region_order_price, rs_price['Median_Price'],
              color=colors_l, edgecolor='white', linewidth=0.8)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + rs_price['Median_Price'].max() * 0.01,
            f'${bar.get_height()/1000:.0f}K',
            ha='center', fontsize=9.5, fontweight='bold')
ax.set_title('Median Resale Price by Region', fontsize=11)
ax.set_ylabel('Median Resale Price (SGD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_ylim(0, rs_price['Median_Price'].max() * 1.18)

# ── Right: Price per sqm (controls for flat size) ────────────
ax = axes[1]
rs_ppsqm  = region_stats.set_index('region').reindex(region_order_ppsqm)
colors_r  = get_region_colors(region_order_ppsqm)
bars2 = ax.bar(region_order_ppsqm, rs_ppsqm['Median_Price_per_sqm'],
               color=colors_r, edgecolor='white', linewidth=0.8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + rs_ppsqm['Median_Price_per_sqm'].max() * 0.01,
            f'${bar.get_height():,.0f}',
            ha='center', fontsize=9.5, fontweight='bold')
ax.set_title('Median Price per sqm by Region\n(controls for flat size)', fontsize=11)
ax.set_ylabel('Median Price per sqm (SGD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylim(0, rs_ppsqm['Median_Price_per_sqm'].max() * 1.18)

# Region colour legend
legend_handles = [mpatches.Patch(color=v, label=k) for k, v in REGION_COLORS.items()
                  if k in region_stats['region'].values]
axes[1].legend(handles=legend_handles, loc='upper right', fontsize=8.5, title='Region')

plt.suptitle(
    f'Figure 4: Does the Central Region Command a Price Premium? ({START_YEAR}–{END_YEAR})',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('charts/Figure4_Q4_RegionalPremium.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved → charts/Figure4_Q4_RegionalPremium.png')

### 📝 Q4 Written Findings

---

**Finding:**  
The **Central** region commands the highest median resale price at **$[___]** 

**[North-East / East / West / North]** region offers the best balance of price and accessibility, with median prices of **$[___]** and good MRT connectivity.

**Business Recommendation:**  
The data confirms that the Central region commands a consistent and significant premium over all other regions — primarily driven by proximity to employment hubs, transport networks, and amenities. Buyers who work in the CBD but can tolerate a **30–45 minute commute** would achieve materially better value in the **[recommended region]** region, potentially saving **$[___]** or more on a comparable flat.

---